# Predictive Maintenance AI — Exploratory Data Analysis
Author: **Pankaj Kumar**

This notebook provides an in-depth exploratory analysis of multivariate industrial sensor telemetry inspired by the NASA CMAPSS benchmark. We examine degradation signatures across 14 sensor channels, analyze operational regime shifts, and investigate correlations with Remaining Useful Life (RUL).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to system path
sys.path.insert(0, str(Path.cwd().parent))
from src.data.generate_dataset import generate_fleet_data
from src.utils.preprocessing import SENSOR_COLS

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. Dataset Generation & Loading
Let us inspect a sample fleet of machines to observe run-to-failure cycles.

In [ ]:
df = generate_fleet_data(n_units=20, seed=42)
print(f"Total Records: {len(df):,}")
print(f"Equipment Units: {df['unit_id'].nunique()}")
print(f"Average Lifetime: {df.groupby('unit_id')['cycle'].max().mean():.1f} cycles")
df.head()

## 2. Sensor Trajectories Over Machine Lifespan
Observe how mechanical wear accelerates degradation indicators like bearing temperature, vibration, and motor current.

In [ ]:
unit_1 = df[df['unit_id'] == 1]
key_sensors = ['vibration_x', 'vibration_y', 'temperature_bearing', 'pressure_out', 'current_draw', 'acoustic_emission']

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
axes = axes.flatten()

for i, sensor in enumerate(key_sensors):
    axes[i].plot(unit_1['cycle'], unit_1[sensor], lw=1.8, color='#1f77b4')
    axes[i].axvline(x=unit_1['cycle'].max() - 30, color='red', linestyle='--', alpha=0.7, label='Failure Imminent (<=30 cycles)')
    axes[i].set_title(f'Sensor: {sensor}')
    axes[i].set_ylabel('Reading')
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend(loc='upper left')

axes[-1].set_xlabel('Operating Cycle')
axes[-2].set_xlabel('Operating Cycle')
plt.tight_layout()
plt.show()

## 3. Feature Correlation with Remaining Useful Life (RUL)
Computing correlation values reveals which sensors have the strongest linear and monotonic trends with degradation.

In [ ]:
corrs = df[SENSOR_COLS].apply(lambda s: s.corr(df['RUL'])).sort_values()
plt.figure(figsize=(10, 5))
colors = ['#d62728' if c < 0 else '#2ca02c' for c in corrs]
corrs.plot(kind='barh', color=colors)
plt.title('Sensor Correlation with Remaining Useful Life (RUL)')
plt.xlabel('Pearson Correlation')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Key Takeaways
- Severe degradation is characterized by non-linear exponential rises in vibration and thermal parameters.
- Dual-head model design simultaneously estimates continuous RUL and triggers an imminent failure warning threshold.